### Задание 1(Вариант 1)
Даны независимые выборки $X_1,\dots,X_n \sim \mathcal N(\mu_1,\sigma_1^2)$ и $Y_1,\dots,Y_m \sim \mathcal N(\mu_2,\sigma_2^2)$ с известными дисперсиями. Построить доверительный интервал для $\tau=\mu_1-\mu_2$ и исследовать его при разных $n,m$.

В эксперименте: $\mu_1=2$, $\mu_2=1$, $\sigma_1^2=1$, $\sigma_2^2=0.5$, поэтому $\tau=1$; $\alpha=0.05$, число повторений — $10000$.

### Построение интервала
Используем оценку $\widehat\tau=\overline X-\overline Y$. По нормальности и независимости:
$$
\widehat\tau\sim\mathcal N\left(\tau,\frac{\sigma_1^2}{n}+\frac{\sigma_2^2}{m}\right)
$$
$$
Z=\frac{\widehat\tau-\tau}{\sqrt{\sigma_1^2/n+\sigma_2^2/m}}
\sim\mathcal N(0,1)
$$

Пусть $z_{1-\alpha/2}=\Phi^{-1}(1-\alpha/2)$. Тогда
$$
P\left(-z_{1-\alpha/2}\le Z\le z_{1-\alpha/2}\right)=1-\alpha
$$

Отсюда точный доверительный интервал уровня $0.95$:
$$
[L,U]=\left[
\widehat\tau-r,\;
\widehat\tau+r
\right]
$$
$$
r=\Phi^{-1}(0.975)\sqrt{\frac1n+\frac{0.5}{m}}
$$

In [3]:
import numpy as np
from scipy.stats import norm
import math

rng = np.random.default_rng(123)
mu1 = 2
mu2 = 1
tau = mu1 - mu2
sigma1 = 1
sigma2 = math.sqrt(0.5)
numberOfExp = 10000
alpha = 0.05

def exp1(n, m):
    samples1 = rng.normal(mu1, sigma1, (numberOfExp, n))
    samples2 = rng.normal(mu2, sigma2, (numberOfExp, m))
    m1 = samples1.mean(axis=1)
    m2 = samples2.mean(axis=1)
    estimate = m1 - m2
    r = norm.ppf(1 - alpha / 2) * math.sqrt(sigma1 ** 2 / n + sigma2 ** 2 / m)
    lowerBounds = estimate - r
    upperBounds = estimate + r
    cov = np.mean((lowerBounds <= tau) & (tau <= upperBounds))
    width = np.mean(upperBounds - lowerBounds)
    print(f"{n:5d} {m:5d} {cov:10.4f} {lowerBounds.mean():13.6f} {upperBounds.mean():13.6f} {width:13.6f}")

groups = {"n = m": [(10, 10), (50, 50), (100, 100), (500, 500), (1000, 1000)], 
          "n + m = fixed": [(10, 990), (100, 900), (500, 500), (586, 414), (900, 100), (990, 10)]}

for name, sizes in groups.items():
    print(f"\ngroup: {name}")
    print("-"*73)
    print(f"{'n':>5} {'m':>5} {'coverage':>10} {'mean lower':>13} {'mean upper':>13} {'mean width':>13}")
    print("-"*73)
    for n, m in sizes:
        exp1(n, m)



group: n = m
-------------------------------------------------------------------------
    n     m   coverage    mean lower    mean upper    mean width
-------------------------------------------------------------------------
   10    10     0.9537      0.245076      1.763258      1.518182
   50    50     0.9442      0.660577      1.339528      0.678951
  100   100     0.9511      0.763146      1.243238      0.480091
  500   500     0.9539      0.893032      1.107735      0.214703
 1000  1000     0.9511      0.924101      1.075919      0.151818

group: n + m = fixed
-------------------------------------------------------------------------
    n     m   coverage    mean lower    mean upper    mean width
-------------------------------------------------------------------------
   10   990     0.9531      0.383690      1.626406      1.242716
  100   900     0.9512      0.798240      1.200974      0.402734
  500   500     0.9517      0.892824      1.107527      0.214703
  586   414     0.

### Как проводился эксперимент
Для каждой пары $(n,m)$ генерировались `numberOfExp = 10000` независимых пар выборок.

Далее вычислялись выборочные средние `m1`, `m2`, оценка разности `estimate = m1 - m2`.

Радиус интервала рассчитывался как

`r = norm.ppf(1 - alpha / 2) * math.sqrt(sigma1 ** 2 / n + sigma2 ** 2 / m)`.

В таблицу выводились:
- `coverage` — доля интервалов, содержащих истинное значение `tau = 1`
- `mean lower` и `mean upper` — средние границы
- `mean width` — средняя ширина интервала

Сравнивались пары из групп `n = m` и `n + m = 1000`. Использовался сид `123`.

### Интерпретация результатов
Во всех экспериментах доля интервалов, содержащих истинное значение `tau = 1`, как и ожидалось близка к заданному уровню $0.95$.

При `n = m` увеличение объёмов с `10` до `1000` уменьшает ширину интервала примерно в `10` раз с `1.5182` до `0.1518`, это объясняется тем что ширина примерно равна $2\cdot1.96\sqrt{1.5/n}$ и убывает как $1/\sqrt n$

При `n + m = 1000` минимальный интервал достигается при `n, m = (586, 414)`. Доказательство: радиус зависит от суммы $\frac1n+\frac{0.5}{m}$. Найдём минимум этой функции при $n+m=1000$, подставим $m=1000-n$:
$$
f(n)=\frac1n+\frac{0.5}{1000-n}
$$

$$
f'(n)=-\frac1{n^2}+\frac{0.5}{(1000-n)^2}=0
$$

$$
n_*=\frac{1000\sqrt2}{1+\sqrt2}\approx586.78,
\qquad
m_*=\frac{1000}{1+\sqrt2}\approx414.21
$$

Значит минимальная ширина интервала достигается при $(n,m)=(586,414)$. Это подтверждается результатом эксперимента.

Также для `(900, 100)` интервал уже, чем для `(100, 900)`. Это объясняется тем, что первая выборка имеет большую дисперсию, поэтому увеличение её объёма сильнее снижает дисперсию оценки.

Таким образом, изменение объёмов выборок влияет на точность оценивания, а покрытие остаётся близким к $0.95$.